# Guion de Video — Proyecto HTTP (20–25 min)

## Persona A — Primera parte completa (0:00–17:30)

### 0:00–1:30 Presentación y objetivos
- Presentación: Servidor HTTP/1.0 en Rust con enrutamiento, pools de workers (basic/cpu/io), métricas detalladas y sistema de jobs con persistencia ligera.
- Agenda: estructura de `src`, puntos clave del código, cómo correrlo y demo con `curl`.

### 1:30–4:30 Estructura del proyecto (`src`)
- `main.rs`: arranque, carga de `Config`, construcción de `Router`, `AppState::shared`, `spawn_dispatcher` y `http_listen_loop`.
- `core.rs`: núcleo HTTP (parseo del request, enrutamiento, encolado en pool y escritura de respuesta), `AppState`, serialización HTTP, métricas.
- `router.rs`: mapeo de rutas → pool y handler.
- `config.rs`: variables de entorno para puerto, workers y profundidad de colas, límites y timeouts.
- `workers.rs`: colas con backpressure y pools de workers, flags `busy` por worker.
- `metrics.rs`: métricas globales y por comando (wait/exec, avg, stddev y p50/p95/p99).
- `jobs.rs`: job store con journal JSONL, scheduler con límites por tipo y timeouts; reusa handlers.
- `handlers/`: endpoints separados en `basic.rs`, `cpu.rs`, `io.rs`, `jobs.rs`.

### 4:30–8:00 Arranque en `main.rs` y estado global
- Flujo: `Config::from_env_or_default()` → `Router::new()` → `AppState::shared(cfg, router)` → `spawn_dispatcher(shared)` → `http_listen_loop(&shared)`.
- `AppState::shared` en dos pasos: crea un `dummy` (con `Pools::new_dummy()`), luego construye `Pools::new(&dummy)` y devuelve `AppState` definitivo con pools reales.
- Beneficio: tener `Shared = Arc<AppState>` disponible al construir pools.

### 8:00–11:30 Router y manejo de conexión (visión general)
- `router.rs`: cada ruta se mapea a `Route::{Basic|Cpu|Io}` con una función handler.
- `core::handle_connection`: parsea la request-line (solo GET), arma `Request`, consulta al router y encola la tarea en el pool correspondiente. Si no existe, 404.
- `enqueue_on_pool`: captura tiempos `wait_ms` y `exec_ms`, registra métricas por comando y maneja backpressure (503 con `Retry-After`).

### 11:30–14:30 Pools y backpressure (conceptos)
- `WorkQueue`: canal MPSC + contador `pending` con `Mutex`, límite `max_depth` y `workers` hilos. Cada worker marca `busy` con `AtomicBool`.
- `submit`: si `pending >= max_depth` → error de backpressure con sugerencia de reintento.
- `Pools::new`: usa `Config` para tamaños de cola y número de workers por pool.

### 14:30–17:30 Métricas y endpoints de inspección
- Métricas globales: `accepted`, `handled`, `errors`, latencias globales y muestras.
- Métricas por comando: colecciones de muestras `wait_ms` y `exec_ms`, cálculo de avg/stddev/p50/p95/p99.
- Endpoints:
  - `/status`: snapshot de colas, workers por pool, config y timeouts.
  - `/metrics`: estadísticos por comando (latencia y colas), estimación de workers ocupados y throughput.

### 17:30–22:00 (Transición a Persona B)
- Cierre de mi parte: Arquitectura desde arriba hacia abajo lista. Ahora veremos handlers (basic/cpu/io), sistema de jobs en doble modo y demo completa.

---

## Apéndice rápido para Persona A (comandos de ejecución)

```bash
# PowerShell en Windows
cd C:\Github\ServidorHTTP\proyecto-http
$env:PORT="8080"           # opcional
$env:WORKERS_CPU="4"       # opcional
cargo run
```


## Persona B — Segunda parte completa (17:30–fin)

### 17:30–20:00 Handlers `basic.rs` (demo rápida)
- Puntos: Firma `(state, req) -> (status, content_type, body_bytes)`, validaciones y mensajes de error claros.
- Demos:
  - `GET /timestamp`
  - `GET /reverse?text=hola`
  - `GET /toupper?text=AbCd`
  - `GET /random?count=5&min=10&max=20`
  - `GET /hash?text=hola`
- Archivos: `createfile`/`deletefile` con validación de nombre (sin `..` ni separadores) y códigos 409/500 según caso.

Comandos de ejemplo:
```bash
curl http://localhost:8080/timestamp
curl "http://localhost:8080/reverse?text=hola"
curl "http://localhost:8080/toupper?text=AbCd"
curl "http://localhost:8080/random?count=5&min=10&max=20"
curl "http://localhost:8080/hash?text=hola"
```

### 20:00–24:00 Handlers `cpu.rs` y `io.rs` + doble modo job
- Doble modo en CPU/IO: `?mode=job&prio=low|normal|high` encola el trabajo y devuelve `job_id`; sin ese parámetro, ejecuta directo.
- CPU-bound ejemplos:
  - `GET /isprime?n=97&method=auto`
  - `GET /factor?n=360`
  - `GET /pi?digits=10`
  - `GET /mandelbrot?width=20&height=20&max_iter=100`
  - `GET /matrixmul?size=20&seed=123`
- IO-bound de archivos (lee desde `data/`):
  - `GET /sortfile?name=nums.txt&algo=merge|quick`
  - `GET /wordcount?name=nums.txt`
  - `GET /grep?name=nums.txt&pattern=3`
  - `GET /compress?name=nums.txt&codec=gzip|xz`
  - `GET /hashfile?name=nums.txt&algo=sha256`

Comandos de ejemplo:
```bash
curl "http://localhost:8080/isprime?n=97&method=auto"
curl "http://localhost:8080/factor?n=360"
curl "http://localhost:8080/pi?digits=10"
curl "http://localhost:8080/mandelbrot?width=20&height=20&max_iter=100"
curl "http://localhost:8080/matrixmul?size=20&seed=123"

# Crear y usar archivo
curl "http://localhost:8080/createfile?name=nums.txt&content=5,3,4,1,2&repeat=1"
curl "http://localhost:8080/sortfile?name=nums.txt&algo=merge"
curl "http://localhost:8080/wordcount?name=nums.txt"
curl "http://localhost:8080/grep?name=nums.txt&pattern=3"
curl "http://localhost:8080/compress?name=nums.txt&codec=gzip"
curl "http://localhost:8080/hashfile?name=nums.txt"
curl "http://localhost:8080/deletefile?name=nums.txt"
```

### 24:00–27:00 Sistema de Jobs (`jobs.rs`)
- Persistencia efímera: journal JSONL en `data/jobs.journal`. Recuperación al arranque y saneo de jobs `Running` → `Error("restarted")`.
- `spawn_dispatcher`: scheduler en hilo aparte:
  - Ordena pendientes por prioridad y FIFO.
  - Respeta límites de concurrencia por tipo (CPU/IO) desde `Config`.
  - Ejecuta cada job en hilo, aplica `recv_timeout` con `timeout_ms` por tarea.
- Endpoints:
  - `GET /jobs/submit?task=...&<params>&prio=...`
  - `GET /jobs/status?id=job-XXXX`
  - `GET /jobs/result?id=job-XXXX`
  - `GET /jobs/cancel?id=job-XXXX`

Comandos de ejemplo (job mode y API jobs):
```bash
# Encolar por doble modo
a=$(curl -s "http://localhost:8080/isprime?n=999983&mode=job&prio=high")
# o vía submit
a=$(curl -s "http://localhost:8080/jobs/submit?task=isprime&n=999983&prio=high")
JOB=$(echo $a | sed -E 's/.*"job_id":"([^"]+)".*/\1/')

curl "http://localhost:8080/jobs/status?id=$JOB"
curl "http://localhost:8080/jobs/result?id=$JOB"
curl "http://localhost:8080/jobs/cancel?id=$JOB"
```

### 27:00–29:00 Métricas y backpressure en vivo
- `/metrics`: p50/p95/p99, avg/stddev por comando; colas y estimación de workers ocupados; throughput.
- Backpressure: si una cola se satura (`pending >= max_depth`), respuesta 503 con `Retry-After`. 
- Mostrar cómo influye configuración:
  - Variables: `WORKERS_BASIC|CPU|IO`, `QUEUE_*`, `JOBS_QUEUE_MAX`, `TIMEOUT_CPU_MS`, `TIMEOUT_IO_MS`, `MAX_RUNNING_CPU_JOBS`, `MAX_RUNNING_IO_JOBS`.

Ejemplo rápido:
```bash
# Ajustar para forzar presión
# PowerShell (nueva instancia antes de run)
$env:QUEUE_CPU="1"; $env:WORKERS_CPU="1"
# En otra consola, lanzar múltiples /matrixmul y observar 503 y Retry-After
```

### 29:00–30:00 Cierre y próximos pasos
- Recap: arquitectura simple pero completa (HTTP 1.0, pools, métricas y jobs con persistencia).
- Posibles mejoras: soporte HTTP/1.1 keep-alive, más métodos, parsing robusto, persistencia durable, exposición Prometheus, tests de carga.

---

## Apéndice rápido para Persona B (comandos útiles)

```bash
# Endpoints de ayuda e inspección
curl http://localhost:8080/help
curl http://localhost:8080/status
curl http://localhost:8080/metrics
```


# Explicación del Servidor HTTP - Guion para Video

## 1. Introducción y Objetivo

Hemos desarrollado un servidor HTTP/1.0 completo en Rust que demuestra los conceptos fundamentales de sistemas operativos: **concurrencia, sincronización, planificación y manejo de colas de trabajo**.

El servidor es capaz de:
- Atender múltiples clientes simultáneamente
- Procesar diferentes tipos de tareas (CPU-intensivas y IO-intensivas)
- Manejar trabajos largos de forma asíncrona mediante un sistema de jobs
- Proporcionar métricas detalladas de rendimiento

## 2. Arquitectura General

### 2.1 Diseño por Capas

El servidor está organizado en módulos claramente separados:

- **Router**: Decide qué handler ejecutar según la ruta HTTP
- **Handlers**: Procesan comandos específicos (básicos, CPU, IO)
- **Workers**: Pools especializados de hilos para cada tipo de trabajo
- **Job Manager**: Sistema de colas para tareas largas
- **Métricas**: Recopilación y análisis de rendimiento

### 2.2 Worker Pools Especializados

Tenemos **tres pools de workers** independientes:

1. **Basic Pool**: Para comandos simples y rápidos (status, timestamp, reverse)
2. **CPU Pool**: Para tareas intensivas en procesamiento (isprime, factor, pi, matrixmul)
3. **IO Pool**: Para operaciones de entrada/salida (sortfile, compress, hashfile)

Esta separación permite optimizar el uso de recursos: los trabajos CPU no bloquean las operaciones de disco, y viceversa.

## 3. Funcionalidades Principales

### 3.1 Comandos Básicos

Implementamos todos los comandos básicos del enunciado:
- `/status`: Estado del servidor, uptime, configuración
- `/timestamp`: Fecha y hora actual
- `/reverse`, `/toupper`: Transformaciones de texto
- `/random`: Números aleatorios con rangos
- `/createfile`, `/deletefile`: Gestión de archivos
- `/simulate`, `/loadtest`: Pruebas de carga y rendimiento

### 3.2 Procesamiento CPU-Intensivo

**Comandos implementados:**
- `/isprime?n=N`: Prueba de primalidad con dos algoritmos (división y Miller-Rabin)
- `/factor?n=N`: Factorización completa en números primos
- `/pi?digits=D`: Cálculo de π con precisión configurable
- `/mandelbrot?width=W&height=H&max_iter=I`: Generación de fractales
- `/matrixmul?size=N&seed=S`: Multiplicación de matrices grandes con verificación SHA-256

**Característica clave**: Todos realizan trabajo real, no usan `sleep()` artificial.

### 3.3 Procesamiento IO-Intensivo

**Comandos implementados:**
- `/sortfile?name=FILE&algo=merge|quick`: Ordena archivos grandes (≥50MB soportados)
- `/wordcount?name=FILE`: Cuenta líneas, palabras y bytes (equivalente a `wc`)
- `/grep?name=FILE&pattern=PATTERN`: Búsqueda de patrones en archivos
- `/compress?name=FILE&codec=gzip|xz`: Compresión real con gzip o xz
- `/hashfile?name=FILE&algo=sha256`: Cálculo de hash SHA-256

**Característica clave**: Manejan archivos grandes eficientemente usando streaming y buffers.

## 4. Sistema de Jobs (Job Manager)

### 4.1 ¿Por qué un sistema de jobs?

Las tareas largas (como calcular π con muchos dígitos o ordenar archivos de 50MB+) pueden tardar varios segundos o minutos. En HTTP/1.0, mantener conexiones abiertas tanto tiempo no es práctico.

**Solución**: Sistema de jobs asíncrono.

### 4.2 Funcionamiento

**Modo directo vs modo job:**
- **Modo directo**: `/isprime?n=97` → respuesta inmediata
- **Modo job**: `/isprime?n=6700417&mode=job&prio=high` → devuelve `job_id` inmediatamente

**Flujo completo:**
1. Cliente envía request con `?mode=job`
2. Servidor encola el trabajo y retorna `{ "job_id": "...", "status": "queued" }`
3. Un dispatcher en background toma trabajos de la cola según prioridad
4. Cliente consulta `/jobs/status?id=JOBID` para ver progreso
5. Cuando termina, cliente obtiene resultado con `/jobs/result?id=JOBID`

### 4.3 Características del Job Manager

- **Prioridades**: `low`, `normal`, `high` (FIFO dentro de cada prioridad)
- **Límites de concurrencia**: Controla cuántos trabajos CPU e IO corren simultáneamente
- **Backpressure**: Si la cola está llena, retorna `503 Service Unavailable`
- **Timeouts**: Trabajos que exceden tiempo máximo son cancelados automáticamente
- **Persistencia efímera**: Metadatos sobreviven a reinicios gracefull (journal)

### 4.4 Endpoints del Sistema de Jobs

- `/jobs/submit?task=TASK&<params>&prio=priority`: Encola un trabajo
- `/jobs/status?id=JOBID`: Estado actual (queued/running/done/error/canceled) y progreso
- `/jobs/result?id=JOBID`: Resultado final en formato JSON del comando
- `/jobs/cancel?id=JOBID`: Cancelación de trabajos en cola o ejecución

## 5. Métricas y Observabilidad

### 5.1 Endpoint `/metrics`

Proporciona estadísticas detalladas por comando:

**Estructura de respuesta:**
```json
{
  "queues": {
    "isprime": 0,
    "sortfile": 2,
    ...
  },
  "workers": {
    "isprime": { "total": 4, "busy": 1 },
    ...
  },
  "latency_ms": {
    "isprime": {
      "count": 150,
      "avg_wait_ms": 12.5,
      "avg_exec_ms": 45.2,
      "stddev_wait_ms": 8.3,
      "stddev_exec_ms": 15.7,
      "p50": 42,
      "p95": 78,
      "p99": 95
    },
    ...
  },
  "throughput": { "requests_per_second": 25 },
  "requests": { "accepted": 1000, "handled": 998, "errors": 2 }
}
```

### 5.2 Métricas Clave

**Por comando capturamos:**
- **Tiempo de espera (wait_ms)**: Cuánto espera un trabajo en cola antes de ser procesado
- **Tiempo de ejecución (exec_ms)**: Tiempo real de procesamiento
- **Estadísticas completas**: Promedio, desviación estándar, percentiles p50/p95/p99

Esto permite identificar cuellos de botella: si `avg_wait_ms` es alto, necesitamos más workers.

### 5.3 Endpoint `/status`

Información general del servidor:
- Estado operativo y puerto
- PID del proceso
- Tiempo activo (uptime)
- Contadores globales (conexiones aceptadas/atendidas/errores)
- Configuración actual (workers, colas, timeouts)
- Estado de colas por pool

## 6. Sincronización y Thread Safety

### 6Σ.1 Uso de Arc<Mutex<...>>

Como requerimiento del curso, usamos `Arc<Mutex<...>>` para compartir estado entre hilos:
- **Metrics**: Todos los contadores y muestras protegidos con Mutex
- **JobStore**: HashMap de jobs con acceso thread-safe
- **Pools**: Colas de trabajos sincronizadas

### 6.2 Evitar Data Races

- Cada worker tiene su propia cola de tareas con acceso exclusivo
- Los contadores globales se incrementan dentro de locks
- Los jobs se clonan antes de pasarse a hilos de ejecución

### 6.3 Sin Deadlocks

- Locks se adquieren en orden consistente
- Timeouts en operaciones bloqueantes
- Separación clara de responsabilidades entre componentes

## 7. Configuración y Escalabilidad

### 7.1 Variables de Entorno

El servidor es altamente configurable:
- `PORT`: Puerto de escucha (default: 8080)
- `WORKERS_BASIC`, `WORKERS_CPU`, `WORKERS_IO`: Número de workers por pool
- `QUEUE_BASIC`, `QUEUE_CPU`, `QUEUE_IO`: Profundidad máxima de colas
- `MAX_RUNNING_CPU_JOBS`, `MAX_RUNNING_IO_JOBS`: Límites de concurrencia para jobs
- `TIMEOUT_CPU_MS`, `TIMEOUT_IO_MS`: Timeouts por tipo de trabajo

### 7.2 Escalabilidad Horizontal

Gracias a los pools especializados, podemos escalar cada tipo de trabajo independientemente:
- Si hay muchos cálculos matemáticos → aumentar `WORKERS_CPU`
- Si hay muchas operaciones de archivo → aumentar `WORKERS_IO`

Esto es más eficiente que un pool único donde un trabajo CPU bloquea trabajos IO.

## 8. Ejemplos de Uso Práctico

### 8.1 Ejecución Directa (Síncrona)

```bash
# Cálculo rápido de primalidad
curl "http://localhost:8080/isprime?n=97"
# Respuesta inmediata: {"n":97,"is_prime":true,"method":"miller-rabin","elapsed_ms":2}

# Ordenar archivo pequeño
curl "http://localhost:8080/sortfile?name=datos.txt&algo=merge"
# Respuesta: {"file":"datos.txt","algo":"merge","sorted_file":"datos.txt.sorted",...}
```

### 8.2 Ejecución Asíncrona (Jobs)

```bash
# Encolar trabajo grande
curl "http://localhost:8080/isprime?n=6700417&mode=job&prio=high"
# Respuesta: {"job_id":"job-1234567890","status":"queued","task":"isprime","priority":"high"}

# Consultar estado
curl "http://localhost:8080/jobs/status?id=job-1234567890"
# Respuesta: {"status":"running","progress":65,"eta_ms":1200}

# Obtener resultado cuando termine
curl "http://localhost:8080/jobs/result?id=job-1234567890"
# Respuesta: {"n":6700417,"is_prime":true,"method":"miller-rabin"}
```

### 8.3 Monitoreo

```bash
# Ver métricas en tiempo real
curl "http://localhost:8080/metrics" | jq

# Ver estado del servidor
curl "http://localhost:8080/status" | jq
```

## 9. Puntos Destacables de la Implementación

### 9.1 Cumplimiento del Enunciado

✅ **Todos los comandos requeridos implementados**  
✅ **Soporte para archivos ≥50MB** (probado con sortfile)  
✅ **Sistema de jobs completo** con submit/status/result/cancel  
✅ **Métricas detalladas** con p50/p95/p99 y estadísticas por comando  
✅ **Backpressure** y gestión de timeouts  
✅ **Persistencia efímera** de jobs (journal)  
✅ **Trazabilidad** con X-Request-Id en todas las respuestas  

### 9.2 Buenas Prácticas

- **Código modular**: Separación clara de responsabilidades
- **Manejo de errores**: Validación de parámetros y mensajes claros
- **Thread safety**: Sin data races ni deadlocks
- **Configurabilidad**: Fácil ajuste sin recompilar
- **Observabilidad**: Métricas comprensibles para debugging y tuning

### 9.3 Desafíos Técnicos Resueltos

1. **Manejo eficiente de archivos grandes**: Streaming y buffers para evitar cargar todo en memoria
2. **Sincronización de métricas**: Uso correcto de Mutex sin degradar rendimiento
3. **Dispatcher de jobs**: Lógica de planificación con prioridades y límites de concurrencia
4. **Algoritmos reales**: Implementación de merge sort, quick sort, pruebas de primalidad, etc.

## 10. Conclusión

Hemos implementado un servidor HTTP completo que demuestra comprensión profunda de:

- **Concurrencia**: Múltiples pools de workers procesando trabajos en paralelo
- **Sincronización**: Uso adecuado de primitivas thread-safe (Mutex, Arc)
- **Planificación**: Sistema de jobs con prioridades y límites de concurrencia
- **Gestión de recursos**: Optimización según tipo de trabajo (CPU vs IO)

El sistema es **robusto, escalable y observable**, proporcionando todas las herramientas necesarias para monitorear y ajustar el rendimiento según la carga de trabajo.

---

**Preparado para demostración en video**  
*Estructura clara para explicación verbal, cubriendo aspectos técnicos sin exceso de detalle*

